# AI3.3 — Qwen3 0.6B LoRA

This notebook fetches the training code from GitHub and runs a **dry run by default**. To train, attach a Kaggle GPU and dataset, set `TRAIN_FILE`, then explicitly change `START_TRAINING` to `True`. Replit does not control Kaggle sessions.

For a private repository, use an authorized GitHub/Kaggle connection or Kaggle's secure secret storage. Never paste credentials into cells, URLs, or notebook output.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Replace the placeholder with the repository's normal GitHub clone URL.
REPO_URL = "https://github.com/YOUR_GITHUB_USER/ai3.3.git"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/ai3.3")

if not (REPO_DIR / "training" / "requirements-kaggle.txt").is_file():
    if "YOUR_GITHUB_USER" in REPO_URL:
        raise ValueError("Set REPO_URL to your GitHub repository clone URL first.")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

print("Repository is ready:", REPO_DIR)

In [ ]:
# Kaggle supplies its own CUDA-enabled PyTorch; this requirements file leaves torch untouched.
requirements = REPO_DIR / "training" / "requirements-kaggle.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)

## Set your dataset and run mode

Attach your dataset through Kaggle's **Add Input** control. Set `TRAIN_FILE` to its JSONL path. Keep `START_TRAINING = False` for a preflight check; set it to `True` only when you are ready to spend GPU time.

In [ ]:
TRAIN_FILE = ""  # Example: /kaggle/input/my-dataset/train.jsonl
EVAL_FILE = ""   # Optional separate JSONL validation file
START_TRAINING = False

training_dir = REPO_DIR / "training"
command = [
    sys.executable,
    "-m",
    "ai33_training.train",
    "--config",
    "configs/qwen3_0_6b_lora.json",
]

if TRAIN_FILE:
    command.extend(["--train-file", TRAIN_FILE])
if EVAL_FILE:
    command.extend(["--eval-file", EVAL_FILE])

if START_TRAINING:
    if not TRAIN_FILE:
        raise ValueError("Set TRAIN_FILE to your attached dataset before training.")
    command.append("--run-training")
else:
    command.append("--dry-run")

environment = os.environ.copy()
source_dir = str(training_dir / "src")
environment["PYTHONPATH"] = source_dir + os.pathsep + environment.get("PYTHONPATH", "")
subprocess.run(command, cwd=training_dir, env=environment, check=True)

## Evaluate a saved adapter (optional)

After training completes, set `ADAPTER_DIR` to the adapter directory and `EVAL_FILE` to a validation JSONL file, then run this cell. Evaluation also requires a Kaggle GPU.

In [ ]:
ADAPTER_DIR = ""  # Example: /kaggle/working/ai3.3/training/outputs/qwen3-0.6b-lora/adapter
if ADAPTER_DIR:
    if not EVAL_FILE:
        raise ValueError("Set EVAL_FILE before running standalone evaluation.")
    eval_command = [
        sys.executable,
        "-m",
        "ai33_training.evaluate",
        "--config",
        "configs/qwen3_0_6b_lora.json",
        "--checkpoint",
        ADAPTER_DIR,
        "--eval-file",
        EVAL_FILE,
    ]
    subprocess.run(eval_command, cwd=training_dir, env=environment, check=True)
else:
    print("Evaluation skipped. Set ADAPTER_DIR and EVAL_FILE when ready.")